# تنزيل حزمة التحكيم البشري — NASN Task 05

هذه حزمة 213 حالة تختلف فيها المراجعتان المصححتان. 150 تختلف في القرارات
الأساسية، و63 إضافية تختلف فقط في عبارة النفي أو وصف نطاقه. اختلاف الصياغة
لا يثبت اختلاف المعنى؛ يُراجع بشريًا ولا يُدمج تلقائيًا.

اختر CPU وشغّل الخلايا بالترتيب، واربط Drive ثم اضغط زر تنزيل حزمة التحكيم.
لا تحتاج GPU أو checkpoint. القرارات الجديدة فارغة؛ لا تُجرى مراجعة أو تحكيم
آلي، ولا يُستخدم LLM لحسم الخلاف. لا يتم تدريب نموذج أو قراءة الاختبار الرسمي.
لم يُنفَّذ هذا الدفتر في Colab هنا؛ تم اختبار التوليد محليًا.

أعطِ الحزمة لشخص ثالث مختلف عن R1 وR2. زوّده أيضًا بملفي A وB المصحّحين
ليقرأ تعليل المراجعين عند التحكيم، دون ملف LLM أو تنبؤات MARBERT. يُسجّل
قراره وتعليله وهوية المراجع ووقت المراجعة. يعود إليك ملف CSV المكتمل لجميع
الحالات الـ213. لا تتغيّر قرارات المراجعتين الأصليتين.


In [ ]:
from pathlib import Path
import os, sys, uuid, datetime, subprocess, json, hashlib, urllib.request
from google.colab import drive
drive.mount('/content/drive')
RUN_ID='adjudication-download-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
REVIEW_ROOT=Path('/content/drive/MyDrive/NASN/task05')/RUN_ID
REVIEW_ROOT.mkdir(parents=True,exist_ok=False)
REVIEW_REPO=Path('/content')/('nasn-'+RUN_ID)
REVIEW_CODE_REF='d3658b4da959913530f77a26f6b3aa115b849493'
def review_call(args, **kwargs):
    result=subprocess.run([str(x) for x in args],capture_output=True,text=True,**kwargs)
    if result.stdout:print(result.stdout,end='')
    if result.stderr:print(result.stderr,end='')
    if result.returncode:
        (REVIEW_ROOT/('failure-'+uuid.uuid4().hex[:8]+'.json')).write_text(json.dumps({'command':[str(x) for x in args],'stdout':result.stdout,'stderr':result.stderr},indent=2))
        result.check_returncode()
    return result
review_call(['git','clone','--no-checkout','https://github.com/haneenbiate/-.git',REVIEW_REPO])
review_call(['git','-C',REVIEW_REPO,'checkout','--detach',REVIEW_CODE_REF])
assert subprocess.check_output(['git','-C',str(REVIEW_REPO),'rev-parse','HEAD'],text=True).strip()==REVIEW_CODE_REF
print('سيتم حفظ المراجعة في:',REVIEW_ROOT)


In [ ]:
# Minimal Python environment: no torch, transformers, GPU or model download.
bootstrap=Path('/content')/('bootstrap-'+RUN_ID)
review_call([sys.executable,'-m','pip','install','--target',bootstrap,'uv==0.13.0'])
uv=bootstrap/'bin/uv'
bootenv={**os.environ,'UV_PYTHON_INSTALL_DIR':'/content/nasn-review-python','UV_CACHE_DIR':'/content/nasn-review-cache','UV_PYTHON_BIN_DIR':'/content/nasn-review-bin'}
review_call([uv,'python','install','--system-certs','3.12.14'],env=bootenv)
managed=subprocess.check_output([str(uv),'python','find','--managed-python','3.12.14'],env=bootenv,text=True).strip()
venv=Path('/content')/('venv-'+RUN_ID)
review_call([managed,'-m','venv',venv])
REVIEW_PY=venv/'bin/python'
assert subprocess.check_output([str(REVIEW_PY),'-c','import platform;print(platform.python_version())'],text=True).strip()=='3.12.14'
review_call([REVIEW_PY,'-m','pip','install','-r',REVIEW_REPO/'requirements.txt',
    'Jinja2==3.1.6','MarkupSafe==3.0.3','openpyxl==3.1.5','et_xmlfile==2.0.0',
    'matplotlib==3.10.8','contourpy==1.3.3','cycler==0.12.1','fonttools==4.61.1',
    'kiwisolver==1.4.9','pillow==12.3.0','pyparsing==3.3.2'])
check=review_call([REVIEW_PY,'-m','pip','check'])
(REVIEW_ROOT/'dependency_check.txt').write_text(check.stdout+check.stderr)
(REVIEW_ROOT/'dependency_freeze.txt').write_text(subprocess.check_output([str(REVIEW_PY),'-m','pip','freeze'],text=True))
REVIEW_ENV={**os.environ,'PYTHONPATH':str(REVIEW_REPO),'MPLBACKEND':'Agg','MPLCONFIGDIR':'/tmp/nasn-review-matplotlib','XDG_CACHE_HOME':'/tmp/nasn-review-cache'}


In [ ]:
# Fetch only pinned official training bytes; never fetch official test text.
train=REVIEW_REPO/'data/raw/training_data.csv';train.parent.mkdir(parents=True,exist_ok=True)
url='https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/eb85513e133cfc67e30aca4968fadbe2594fcb42/ArSarcasm-v2/training_data.csv'
train.write_bytes(urllib.request.urlopen(url).read())
assert hashlib.sha256(train.read_bytes()).hexdigest()=='1da727eb763459f436c4e61f52c6a0bc63d5300af8fcc99b5675ae4616dae04a'
# Explicit candidate preparation only, using existing checksum-verified splits.
preparation=REVIEW_ROOT/'task04-preparation'
review_call([REVIEW_PY,'-m','nasn.task04','prepare','--output',preparation],cwd=REVIEW_REPO,env=REVIEW_ENV)
check_code="""import pandas as pd,sys
from pathlib import Path
root=Path(sys.argv[1]); repo=Path(sys.argv[2])
columns=['variant_id','source_row_id','original_sha256','perturbed_sha256','rule','intensity']
actual=pd.read_csv(root/'human_review_queue.csv',dtype=str,keep_default_na=False)
original=pd.read_csv(repo/'reports/task04/human_review_queue.csv',dtype=str,keep_default_na=False)
assert len(actual)==404 and actual[columns].equals(original[columns]), 'Original review queue changed'
print('Verified original 404 assignments; no human decisions generated')
"""
review_call([REVIEW_PY,'-c',check_code,preparation,REVIEW_REPO],cwd=REVIEW_REPO,env=REVIEW_ENV)


In [ ]:
downloads=REVIEW_ROOT/'adjudication-download'
review_call([REVIEW_PY,'-m','nasn.review_workbooks','export-adjudication',
    '--preparation',preparation,
    '--assignments',REVIEW_REPO/'reports/task05-reviewed/adjudication_assignment.csv',
    '--output',downloads],cwd=REVIEW_REPO,env=REVIEW_ENV)
status=json.loads((downloads/'run_status.json').read_text())
assert status['assigned_n']==213 and status['human_decisions_generated'] is False
assert hashlib.sha256((downloads/'adjudication.zip').read_bytes()).hexdigest()==status['zip_sha256']
print('حزمة التحكيم محفوظة على Drive:',downloads)


## تنزيل الحزمة

اضغط الزر. إذا لم يبدأ التنزيل، افتح Drive ونزّل `adjudication.zip` من المسار
المطبوع أعلاه. بعد فك الضغط افتح `adjudication.html`، وصدّر CSV باستمرار.
لا يصبح العمل مكتملًا حتى تُحسم الخلافات بشريًا وتتوفر تنبؤات Task 04 للتحليل.


In [ ]:
from google.colab import files
import ipywidgets as widgets
from IPython.display import display
button=widgets.Button(description='تنزيل حزمة التحكيم',layout=widgets.Layout(width='240px'))
button.on_click(lambda _:files.download(str(downloads/'adjudication.zip')))
display(button)
